# Lição 12 - Redução do Histórico de Conversa com Bloco de Notas do Agente

Este notebook demonstra como gerenciar o contexto em conversas longas usando o Microsoft Agent Framework. À medida que as conversas crescem, a contagem de tokens aumenta — eventualmente excedendo a janela de contexto do modelo. Abordamos isso com um **padrão de sumarização de contexto** e um **bloco de notas do agente** para memória persistente.

## O que você vai aprender:
1. **Por que o Gerenciamento de Contexto é Importante**: Compreendendo limites de token e janelas de contexto
2. **Agentes Conscientes do Contexto**: Construindo agentes que gerenciam seu próprio contexto de conversa
3. **Padrão de Sumarização de Contexto**: Usando ferramentas para condensar o histórico da conversa
4. **Bloco de Notas do Agente**: Memória persistente que sobrevive à redução de contexto

## Pré-requisitos:
- Configuração do Azure OpenAI com variáveis de ambiente configuradas
- Entendimento dos conceitos básicos de agentes das lições anteriores


## Configuração


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Por que o Gerenciamento de Contexto Importa

Todo LLM tem uma **janela de contexto** finita — o número máximo de tokens que pode processar em uma única solicitação. Conforme uma conversa multi-turno progride:

- **A contagem de tokens cresce linearmente** com cada mensagem do usuário e resposta do assistente.
- **Os tokens do prompt dominam o custo** porque todo o histórico é reenviado a cada interação.
- Eventualmente a conversa **excede a janela de contexto** e o modelo ou trunca ou gera erro.

### Estratégias para Gerenciar o Contexto

| Estratégia | Como Funciona | Compromisso |
|---|---|---|
| **Truncamento** | Descartar as mensagens mais antigas | Perde o contexto inicial |
| **Sumarização** | Condensar mensagens antigas em um resumo | Alguns detalhes são perdidos, mas pontos chave são mantidos |
| **Bloco de Notas / Memória Externa** | Armazenar fatos importantes fora da conversa | Requer chamadas a ferramentas, mas sobrevive a qualquer redução |

Neste notebook combinamos a **sumarização** com uma **ferramenta de bloco de notas** para que o agente possa manter a continuidade mesmo quando o histórico da conversa é condensado.


## Criando um Agente Sensível ao Contexto


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Simulando uma Conversa Longa

Vamos passar por uma conversa de múltiplas trocas para ver como o contexto se acumula. O agente deve reter detalhes chave (preferências, orçamento, datas de viagem) ao longo das interações e demonstrar continuidade.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Observe como o agente retém o contexto das interações anteriores — ele sabe sobre o Japão, sushi, templos, fotografia, o orçamento de $3000, viagem solo e o período de abril. Em uma conversa curta isso funciona bem, mas à medida que a conversa cresce, reenviar todo o histórico se torna custoso.

Vamos continuar a conversa com mais interações para observar o acúmulo de contexto:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Padrão de Resumo de Contexto

À medida que a conversa cresce, podemos usar uma **ferramenta de resumo** para condensar o contexto acumulado em um formato compacto. O agente chama essa ferramenta para registrar as preferências principais de forma que, mesmo que mensagens antigas sejam descartadas, a informação essencial seja preservada.

Este padrão é o bloco de construção para uma redução mais sofisticada do histórico:
1. O agente identifica fatos-chave da conversa
2. Ele chama a ferramenta de resumo para persistí-los
3. Mensagens antigas podem ser removidas com segurança porque o resumo captura o que importa

Abaixo definimos uma ferramenta `summarize_preferences` que o agente pode chamar para registrar um resumo compacto do que aprendeu.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Resumo

Nesta lição você aprendeu como gerenciar contexto em conversas de agentes de longa duração usando o Microsoft Agent Framework:

### Conceitos Principais
- **Janelas de contexto são finitas** — cada token no histórico da conversa tem custo e conta para o limite.
- **Ferramentas de sumarização** permitem que o agente condense o contexto acumulado em resumos compactos, reduzindo o uso de tokens enquanto preserva informações essenciais.
- **Blocos de anotações do agente** fornecem memória externa persistente que sobrevive a qualquer redução da conversa.

### O Que Você Construiu
- Um **agente consciente do contexto** que mantém continuidade em conversas de múltiplas etapas
- Uma **ferramenta de sumarização** (`summarize_preferences`) que registra detalhes principais do usuário em formato compacto
- Uma **conversa de múltiplas etapas** demonstrando retenção de contexto e manejo de mudanças

### Aplicações no Mundo Real
- **Bots de Atendimento ao Cliente**: lembram preferências durante longas sessões de suporte
- **Assistentes Pessoais**: acompanham projetos em andamento sem necessidade de reexplicar o contexto
- **Tutores Educacionais**: mantêm o progresso do aluno ao longo de muitas interações

### Próximos Passos
- Implementar uma ferramenta completa de bloco de anotações com persistência baseada em arquivos
- Adicionar truncamento automático do histórico após sumarização
- Combinar com bancos de dados vetoriais para busca semântica de memória
- Construir agentes que possam retomar conversas dias depois com o contexto completo


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:  
Este documento foi traduzido utilizando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos para garantir a precisão, esteja ciente de que traduções automatizadas podem conter erros ou imprecisões. O documento original em seu idioma nativo deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se a tradução profissional realizada por um humano. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
